In [1]:
import json
import gzip
from pathlib import Path

def load_uniprot_json(path):
    path = Path(path)

    if path.suffix == ".gz":
        with gzip.open(path, "rt", encoding="utf-8") as f:
            data = json.load(f)
    else:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

    if isinstance(data, dict) and "results" in data:
        return data["results"]

    if isinstance(data, list):
        return data

    raise ValueError(f"Unexpected UniProt JSON format: {path}")


pep_records = load_uniprot_json("uniprot_pep.json")
propep_records = load_uniprot_json("uniprot_propep.json")

records_by_acc = {}

for record in pep_records + propep_records:
    acc = record.get("primaryAccession")
    if acc is None:
        print("Warning: record without primaryAccession, skipping")
        continue

    # если запись встретилась второй раз, оставляем первую
    # обычно записи будут идентичны, потому что UniProt отдаёт весь entry
    records_by_acc[acc] = record

records = list(records_by_acc.values())

print("Unique proteins:", len(records))

Unique proteins: 18091


### Filter by pep/propep length [5, 50]

In [2]:
import pandas as pd


MIN_FEATURE_LEN = 5
MAX_FEATURE_LEN = 50

TARGET_FEATURES = {
    "Peptide": "PEPTIDE",
    "Propeptide": "PROPEP",
}


def get_location_value(feature, side):
    """
    side: "start" or "end"
    Returns coordinate value or None.
    UniProt coordinates are 1-based inclusive.
    """
    return (
        feature.get("location", {})
        .get(side, {})
        .get("value")
    )


def get_location_modifier(feature, side):
    return (
        feature.get("location", {})
        .get(side, {})
        .get("modifier")
    )


feature_rows = []

for record in records:
    accession = record.get("primaryAccession")
    entry_name = record.get("uniProtkbId")
    sequence = record.get("sequence", {}).get("value")
    sequence_length = record.get("sequence", {}).get("length")
    organism = record.get("organism", {}).get("scientificName")
    taxon_id = record.get("organism", {}).get("taxonId")

    for feature in record.get("features", []):
        raw_type = feature.get("type")

        if raw_type not in TARGET_FEATURES:
            continue

        label = TARGET_FEATURES[raw_type]

        start = get_location_value(feature, "start")
        end = get_location_value(feature, "end")

        start_modifier = get_location_modifier(feature, "start")
        end_modifier = get_location_modifier(feature, "end")

        drop_reasons = []

        if start is None or end is None:
            drop_reasons.append("missing_location")
            feature_len = None
            feature_seq = None
        else:
            start = int(start)
            end = int(end)
            feature_len = end - start + 1

            if sequence is not None:
                feature_seq = sequence[start - 1:end]
            else:
                feature_seq = None

        # Для clean training лучше брать только точные координаты
        if start_modifier not in (None, "EXACT"):
            drop_reasons.append(f"non_exact_start:{start_modifier}")

        if end_modifier not in (None, "EXACT"):
            drop_reasons.append(f"non_exact_end:{end_modifier}")

        if feature_len is not None:
            if feature_len < MIN_FEATURE_LEN:
                drop_reasons.append("too_short")

            if feature_len > MAX_FEATURE_LEN:
                drop_reasons.append("too_long")

            if sequence_length is not None and end > sequence_length:
                drop_reasons.append("location_out_of_bounds")

            if start < 1:
                drop_reasons.append("location_out_of_bounds")

        is_valid_for_training = len(drop_reasons) == 0

        evidences = feature.get("evidences", [])
        evidence_codes = [
            ev.get("evidenceCode")
            for ev in evidences
            if ev.get("evidenceCode")
        ]

        feature_rows.append({
            "accession": accession,
            "entry_name": entry_name,
            "organism": organism,
            "taxon_id": taxon_id,
            "raw_feature_type": raw_type,
            "label": label,
            "start": start,
            "end": end,
            "length": feature_len,
            "feature_sequence": feature_seq,
            "description": feature.get("description"),
            "start_modifier": start_modifier,
            "end_modifier": end_modifier,
            "evidence_codes": evidence_codes,
            "is_valid_for_training": is_valid_for_training,
            "drop_reason": ";".join(drop_reasons),
        })

features_df = pd.DataFrame(feature_rows)

In [3]:
valid_accessions = set(
    features_df.loc[
        features_df["is_valid_for_training"],
        "accession"
    ]
)

filtered_records = [
    record
    for record in records
    if record.get("primaryAccession") in valid_accessions
]

print("Records before:", len(records))
print("Records after:", len(filtered_records))

Records before: 18091
Records after: 13702


In [4]:
import hashlib

OUT_DIR = Path("uniprot_2026/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_FEATURE_LEN = 5
MAX_FEATURE_LEN = 50

TARGET_TYPES = {
    "Peptide": "PEPTIDE",
    "Propeptide": "PROPEP",
}


def sequence_md5(seq: str) -> str:
    return hashlib.md5(seq.encode("utf-8")).hexdigest()


def get_nested(obj, *keys, default=None):
    cur = obj
    for key in keys:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(key)
        if cur is None:
            return default
    return cur


def get_location(feature):
    loc = feature.get("location", {}) or {}

    start_obj = loc.get("start", {}) or {}
    end_obj = loc.get("end", {}) or {}

    start = start_obj.get("value")
    end = end_obj.get("value")

    start_modifier = start_obj.get("modifier")
    end_modifier = end_obj.get("modifier")

    if start is not None:
        start = int(start)

    if end is not None:
        end = int(end)

    return start, end, start_modifier, end_modifier


def get_evidence(feature):
    evidence_codes = []
    evidence_sources = []
    pubmed_ids = []

    for ev in feature.get("evidences", []) or []:
        code = ev.get("evidenceCode")
        source = ev.get("source")
        ev_id = ev.get("id")

        if code:
            evidence_codes.append(code)

        if source:
            evidence_sources.append(source)

        if source == "PubMed" and ev_id:
            pubmed_ids.append(ev_id)

    return evidence_codes, evidence_sources, pubmed_ids


def coords_to_string(features_df: pd.DataFrame, label: str) -> str:
    sub = features_df[features_df["label"] == label].sort_values(["start", "end"])

    if sub.empty:
        return ""

    return ",".join(
        f"{int(row.start)}-{int(row.end)}"
        for row in sub.itertuples(index=False)
    )


protein_rows = []
feature_rows = []

for record in records:
    accession = record.get("primaryAccession")
    entry_name = record.get("uniProtkbId")

    sequence = get_nested(record, "sequence", "value")
    seq_len = get_nested(record, "sequence", "length")

    if not accession or not sequence:
        continue

    organism = get_nested(record, "organism", "scientificName")
    taxon_id = get_nested(record, "organism", "taxonId")
    lineage = get_nested(record, "organism", "lineage", default=[])

    protein_name = get_nested(
        record,
        "proteinDescription",
        "recommendedName",
        "fullName",
        "value",
    )

    gene_names = []
    for gene in record.get("genes", []) or []:
        gene_name = get_nested(gene, "geneName", "value")
        if gene_name:
            gene_names.append(gene_name)

    protein_rows.append({
        "accession": accession,
        "protein_id": accession,
        "entry_name": entry_name,
        "sequence": sequence,
        "sequence_md5": sequence_md5(sequence),
        "length": seq_len if seq_len is not None else len(sequence),
        "organism": organism,
        "taxon_id": taxon_id,
        "lineage": lineage,
        "protein_name": protein_name,
        "gene_names": gene_names,
        "is_viral": "Viruses" in lineage if isinstance(lineage, list) else False,
    })

    for feature in record.get("features", []) or []:
        raw_type = feature.get("type")
        label = TARGET_TYPES.get(raw_type, raw_type)

        start, end, start_modifier, end_modifier = get_location(feature)

        drop_reasons = []
        feature_len = None
        feature_seq = None

        if start is None or end is None:
            drop_reasons.append("missing_location")
        else:
            feature_len = end - start + 1

            if start < 1 or end > len(sequence) or start > end:
                drop_reasons.append("location_out_of_bounds")
            else:
                # UniProt coordinates: 1-based inclusive
                # Python slice: 0-based, end-exclusive
                feature_seq = sequence[start - 1:end]

        if start_modifier not in (None, "EXACT"):
            drop_reasons.append(f"non_exact_start:{start_modifier}")

        if end_modifier not in (None, "EXACT"):
            drop_reasons.append(f"non_exact_end:{end_modifier}")

        is_target_type = raw_type in TARGET_TYPES

        if is_target_type and feature_len is not None:
            if feature_len < MIN_FEATURE_LEN:
                drop_reasons.append("too_short")
            if feature_len > MAX_FEATURE_LEN:
                drop_reasons.append("too_long")

        evidence_codes, evidence_sources, pubmed_ids = get_evidence(feature)

        feature_rows.append({
            "accession": accession,
            "entry_name": entry_name,
            "raw_feature_type": raw_type,
            "label": label,
            "start": start,
            "end": end,
            "length": feature_len,
            "feature_sequence": feature_seq,
            "description": feature.get("description"),
            "start_modifier": start_modifier,
            "end_modifier": end_modifier,
            "evidence_codes": evidence_codes,
            "evidence_sources": evidence_sources,
            "pubmed_ids": pubmed_ids,
            "is_target_type": is_target_type,
            "is_valid_for_training": is_target_type and len(drop_reasons) == 0,
            "drop_reason": ";".join(drop_reasons),
        })


proteins_master = pd.DataFrame(protein_rows).drop_duplicates("accession")
features_master = pd.DataFrame(feature_rows)

valid_features = features_master[
    features_master["is_valid_for_training"]
].copy()

valid_features = valid_features.drop_duplicates(
    subset=["accession", "label", "start", "end"]
)

valid_accessions = set(valid_features["accession"])

proteins_train = proteins_master[
    proteins_master["accession"].isin(valid_accessions)
].copy()

print("Proteins master:", len(proteins_master))
print("Features master:", len(features_master))
print("Valid train proteins:", len(proteins_train))
print("Valid train features:", len(valid_features))
print(valid_features["label"].value_counts())

Proteins master: 18091
Features master: 254566
Valid train proteins: 13702
Valid train features: 20806
label
PEPTIDE    11377
PROPEP      9429
Name: count, dtype: int64


In [5]:
proteins_master.to_parquet(
    OUT_DIR / "proteins_master.parquet",
    index=False,
)

features_master.to_parquet(
    OUT_DIR / "features_master.parquet",
    index=False,
)

### labeled_sequences.csv

In [6]:
legacy_rows = []

for protein in proteins_train.itertuples(index=False):
    acc = protein.accession

    feats = valid_features[
        valid_features["accession"] == acc
    ]

    peptide_coords = coords_to_string(feats, "PEPTIDE")
    propeptide_coords = coords_to_string(feats, "PROPEP")

    legacy_rows.append({
        "protein_id": acc,
        "entry_name": protein.entry_name,
        "sequence": protein.sequence,
        "organism": protein.organism,
        "coordinates": peptide_coords,
        "propeptide_coordinates": propeptide_coords,
    })

labeled_sequences = pd.DataFrame(legacy_rows)

labeled_sequences.to_csv(
    OUT_DIR / "labeled_sequences.csv",
    index=False,
)

print("Saved:", OUT_DIR / "labeled_sequences.csv")
print("Rows:", len(labeled_sequences))
print("With PEPTIDE:", (labeled_sequences["coordinates"] != "").sum())
print("With PROPEP:", (labeled_sequences["propeptide_coordinates"] != "").sum())

Saved: uniprot_2026/processed/labeled_sequences.csv
Rows: 13702
With PEPTIDE: 8181
With PROPEP: 8295


### Fasta files

In [7]:
def write_fasta(df: pd.DataFrame, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for row in df.itertuples(index=False):
            f.write(f">{row.protein_id}\n")
            seq = row.sequence

            for i in range(0, len(seq), 80):
                f.write(seq[i:i + 80] + "\n")


write_fasta(
    labeled_sequences,
    OUT_DIR / "protein_sequences.fasta",
)

print("Saved:", OUT_DIR / "protein_sequences.fasta")

Saved: uniprot_2026/processed/protein_sequences.fasta


In [8]:
report = {
    "n_records_input": len(records),
    "n_proteins_master": int(len(proteins_master)),
    "n_features_master": int(len(features_master)),
    "n_valid_training_features": int(len(valid_features)),
    "n_training_proteins": int(len(labeled_sequences)),

    "valid_feature_counts": {
        str(k): int(v)
        for k, v in valid_features["label"].value_counts().items()
    },

    "all_target_feature_counts": {
        str(k): int(v)
        for k, v in features_master.loc[
            features_master["is_target_type"],
            "label"
        ].value_counts().items()
    },

    "drop_reasons_target_features": {
        str(k): int(v)
        for k, v in features_master.loc[
            features_master["is_target_type"] &
            ~features_master["is_valid_for_training"],
            "drop_reason"
        ].value_counts().items()
    },

    "n_with_peptide": int((labeled_sequences["coordinates"] != "").sum()),
    "n_with_propeptide": int((labeled_sequences["propeptide_coordinates"] != "").sum()),
    "n_with_both": int(
        (
            (labeled_sequences["coordinates"] != "") &
            (labeled_sequences["propeptide_coordinates"] != "")
        ).sum()
    ),
}

with open(OUT_DIR / "build_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "n_records_input": 18091,
  "n_proteins_master": 18091,
  "n_features_master": 254566,
  "n_valid_training_features": 20806,
  "n_training_proteins": 13702,
  "valid_feature_counts": {
    "PEPTIDE": 11377,
    "PROPEP": 9429
  },
  "all_target_feature_counts": {
    "PROPEP": 15287,
    "PEPTIDE": 12381
  },
  "drop_reasons_target_features": {
    "too_long": 3532,
    "too_short": 2501,
    "missing_location;non_exact_start:UNKNOWN": 227,
    "missing_location;non_exact_end:UNKNOWN": 199,
    "non_exact_start:OUTSIDE": 152,
    "non_exact_end:OUTSIDE": 127,
    "non_exact_start:OUTSIDE;too_short": 27,
    "non_exact_start:OUTSIDE;too_long": 22,
    "non_exact_start:UNSURE": 18,
    "non_exact_start:OUTSIDE;non_exact_end:OUTSIDE": 17,
    "non_exact_end:UNSURE;too_long": 10,
    "non_exact_start:UNSURE;non_exact_end:UNSURE;too_long": 9,
    "non_exact_end:OUTSIDE;too_long": 6,
    "non_exact_end:OUTSIDE;too_short": 5,
    "non_exact_start:UNSURE;too_long": 3,
    "non_exact_start:

In [9]:
# Далее запускаем graphpart на labeled_sequences.csv, чтобы разбить на train/val/test с учетом гомологии.

# sudo apt-get update
# sudo apt-get install emboss
# pip install graph-part

# graphpart needle \
#   --fasta-file notebooks/uniprot_2026/processed/protein_sequences.fasta \
#   --threshold 0.3 \
#   --partitions 5 \
#   --out-file notebooks/uniprot_2026/processed/graphpart_assignments.raw.csv \
#   --threads 12 \
#   --no-moving

### Statistics after fold partitioning

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np


ROOT = Path("../data/uniprot_2026")

# при необходимости поправь пути
LABELED_PATH = ROOT / "labeled_sequences.csv"

if not LABELED_PATH.exists():
    LABELED_PATH = ROOT / "processed" / "labeled_sequences.csv"

if not LABELED_PATH.exists():
    raise FileNotFoundError(f"Не найден labeled_sequences.csv в {ROOT}")


def parse_coords(coord_string):
    """
    Input: '12-30,45-60'
    Output: [(12, 30), (45, 60)]
    Coordinates are 1-based inclusive.
    """
    if pd.isna(coord_string):
        return []

    coord_string = str(coord_string).strip()

    if coord_string == "" or coord_string.lower() == "nan":
        return []

    out = []

    for part in coord_string.split(","):
        part = part.strip()

        if not part:
            continue

        start, end = part.split("-")
        out.append((int(start), int(end)))

    return out


def count_residue_labels(sequence, peptide_coords, propeptide_coords):
    """
    Returns residue-level counts.
    Priority: PEPTIDE overwrites PROPEP if overlap exists.
    """
    n = len(sequence)
    labels = np.array(["OTHER"] * n, dtype=object)

    # сначала PROPEP
    for start, end in parse_coords(propeptide_coords):
        labels[start - 1:end] = "PROPEP"

    # потом PEPTIDE с приоритетом
    for start, end in parse_coords(peptide_coords):
        labels[start - 1:end] = "PEPTIDE"

    return {
        "OTHER_residues": int((labels == "OTHER").sum()),
        "PEPTIDE_residues": int((labels == "PEPTIDE").sum()),
        "PROPEP_residues": int((labels == "PROPEP").sum()),
        "total_residues": int(n),
    }


labeled = pd.read_csv(LABELED_PATH)

required_cols = {"protein_id", "sequence", "coordinates", "propeptide_coordinates"}
missing = required_cols - set(labeled.columns)

if missing:
    raise ValueError(f"В labeled_sequences.csv не хватает колонок: {missing}")

# residue counts per protein
count_rows = []

for row in labeled.itertuples(index=False):
    counts = count_residue_labels(
        sequence=row.sequence,
        peptide_coords=row.coordinates,
        propeptide_coords=row.propeptide_coordinates,
    )

    count_rows.append({
        "protein_id": row.protein_id,
        "sequence_length": len(row.sequence),
        "has_peptide": bool(parse_coords(row.coordinates)),
        "has_propeptide": bool(parse_coords(row.propeptide_coordinates)),
        "n_peptide_features": len(parse_coords(row.coordinates)),
        "n_propeptide_features": len(parse_coords(row.propeptide_coordinates)),
        **counts,
    })

protein_counts = pd.DataFrame(count_rows)

# найти все graphpart-файлы
graphpart_files = sorted([
    p for p in ROOT.rglob("graphpart*.csv")
    if "raw" not in p.name.lower()
])

print("Found graphpart files:")
for p in graphpart_files:
    print(" -", p)

if not graphpart_files:
    raise FileNotFoundError(f"Не найдено graphpart*.csv внутри {ROOT}")


def normalize_graphpart(gp):
    """
    Приводим graphpart к колонкам:
    protein_id, cluster
    """
    gp = gp.copy()

    if "protein_id" not in gp.columns:
        if "AC" in gp.columns:
            gp = gp.rename(columns={"AC": "protein_id"})
        elif "accession" in gp.columns:
            gp = gp.rename(columns={"accession": "protein_id"})
        else:
            raise ValueError(f"Не нашёл колонку protein id. Колонки: {gp.columns.tolist()}")

    if "cluster" not in gp.columns:
        raise ValueError(f"Не нашёл колонку cluster. Колонки: {gp.columns.tolist()}")

    gp["cluster"] = gp["cluster"].astype(int)

    return gp[["protein_id", "cluster"]]


all_summaries = {}

for gp_path in graphpart_files:
    gp = normalize_graphpart(pd.read_csv(gp_path))

    df = protein_counts.merge(gp, on="protein_id", how="inner")

    print("\n" + "=" * 80)
    print(gp_path)
    print("labeled proteins:", len(labeled))
    print("assigned proteins:", len(df))
    print("not assigned:", len(labeled) - len(df))

    summary = (
        df.groupby("cluster")
        .agg(
            n_proteins=("protein_id", "count"),
            total_residues=("total_residues", "sum"),
            OTHER_residues=("OTHER_residues", "sum"),
            PEPTIDE_residues=("PEPTIDE_residues", "sum"),
            PROPEP_residues=("PROPEP_residues", "sum"),
            n_with_peptide=("has_peptide", "sum"),
            n_with_propeptide=("has_propeptide", "sum"),
            n_peptide_features=("n_peptide_features", "sum"),
            n_propeptide_features=("n_propeptide_features", "sum"),
        )
        .reset_index()
        .sort_values("cluster")
    )

    for label in ["OTHER", "PEPTIDE", "PROPEP"]:
        summary[f"{label}_frac"] = (
            summary[f"{label}_residues"] / summary["total_residues"]
        )

    total = pd.DataFrame([{
        "cluster": "ALL",
        "n_proteins": df["protein_id"].nunique(),
        "total_residues": df["total_residues"].sum(),
        "OTHER_residues": df["OTHER_residues"].sum(),
        "PEPTIDE_residues": df["PEPTIDE_residues"].sum(),
        "PROPEP_residues": df["PROPEP_residues"].sum(),
        "n_with_peptide": df["has_peptide"].sum(),
        "n_with_propeptide": df["has_propeptide"].sum(),
        "n_peptide_features": df["n_peptide_features"].sum(),
        "n_propeptide_features": df["n_propeptide_features"].sum(),
    }])

    for label in ["OTHER", "PEPTIDE", "PROPEP"]:
        total[f"{label}_frac"] = (
            total[f"{label}_residues"] / total["total_residues"]
        )

    summary_with_total = pd.concat([summary, total], ignore_index=True)

    display_cols = [
        "cluster",
        "n_proteins",
        "total_residues",
        "OTHER_frac",
        "PEPTIDE_frac",
        "PROPEP_frac",
        "n_with_peptide",
        "n_with_propeptide",
        "n_peptide_features",
        "n_propeptide_features",
    ]

    print(summary_with_total[display_cols])

    out_path = gp_path.with_suffix(".class_balance.csv")
    summary_with_total.to_csv(out_path, index=False)

    print("saved:", out_path)

    all_summaries[gp_path.name] = summary_with_total

Found graphpart files:
 - ../data/uniprot_2026/graphpart_assignments.csv

../data/uniprot_2026/graphpart_assignments.csv
labeled proteins: 13702
assigned proteins: 12833
not assigned: 869
  cluster  n_proteins  total_residues  OTHER_frac  PEPTIDE_frac  PROPEP_frac  \
0       0        2536          174852    0.424536      0.334391     0.241072   
1       1        2658          808555    0.898152      0.046550     0.055299   
2       2        2438          653335    0.892311      0.035169     0.072520   
3       3        2674          774384    0.879095      0.052559     0.068346   
4       4        2527           57192    0.202353      0.696513     0.101133   
5     ALL       12833         2468318    0.840955      0.080873     0.078172   

   n_with_peptide  n_with_propeptide  n_peptide_features  \
0            1984               1807                2312   
1            1023               1943                2088   
2             746               2017                1099   
3          

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np


ROOT = Path("../data/uniprot_2022")

# при необходимости поправь пути
LABELED_PATH = ROOT / "labeled_sequences.csv"

if not LABELED_PATH.exists():
    LABELED_PATH = ROOT / "processed" / "labeled_sequences.csv"

if not LABELED_PATH.exists():
    raise FileNotFoundError(f"Не найден labeled_sequences.csv в {ROOT}")


def parse_coords(coord_string):
    """
    Input: '12-30,45-60'
    Output: [(12, 30), (45, 60)]
    Coordinates are 1-based inclusive.
    """
    if pd.isna(coord_string):
        return []

    coord_string = str(coord_string).strip()

    if coord_string == "" or coord_string.lower() == "nan":
        return []

    out = []

    for part in coord_string.split(","):
        part = part.strip().lstrip("(").rstrip(")")

        if not part:
            continue

        start, end = part.split("-")
        out.append((int(start), int(end)))

    return out


def count_residue_labels(sequence, peptide_coords, propeptide_coords):
    """
    Returns residue-level counts.
    Priority: PEPTIDE overwrites PROPEP if overlap exists.
    """
    n = len(sequence)
    labels = np.array(["OTHER"] * n, dtype=object)

    # сначала PROPEP
    for start, end in parse_coords(propeptide_coords):
        labels[start - 1:end] = "PROPEP"

    # потом PEPTIDE с приоритетом
    for start, end in parse_coords(peptide_coords):
        labels[start - 1:end] = "PEPTIDE"

    return {
        "OTHER_residues": int((labels == "OTHER").sum()),
        "PEPTIDE_residues": int((labels == "PEPTIDE").sum()),
        "PROPEP_residues": int((labels == "PROPEP").sum()),
        "total_residues": int(n),
    }


labeled = pd.read_csv(LABELED_PATH)

required_cols = {"protein_id", "sequence", "coordinates", "propeptide_coordinates"}
missing = required_cols - set(labeled.columns)

if missing:
    raise ValueError(f"В labeled_sequences.csv не хватает колонок: {missing}")

# residue counts per protein
count_rows = []

for row in labeled.itertuples(index=False):
    counts = count_residue_labels(
        sequence=row.sequence,
        peptide_coords=row.coordinates,
        propeptide_coords=row.propeptide_coordinates,
    )

    count_rows.append({
        "protein_id": row.protein_id,
        "sequence_length": len(row.sequence),
        "has_peptide": bool(parse_coords(row.coordinates)),
        "has_propeptide": bool(parse_coords(row.propeptide_coordinates)),
        "n_peptide_features": len(parse_coords(row.coordinates)),
        "n_propeptide_features": len(parse_coords(row.propeptide_coordinates)),
        **counts,
    })

protein_counts = pd.DataFrame(count_rows)

# найти все graphpart-файлы
graphpart_files = sorted([
    p for p in ROOT.rglob("graphpart*.csv")
    if "raw" not in p.name.lower()
])

print("Found graphpart files:")
for p in graphpart_files:
    print(" -", p)

if not graphpart_files:
    raise FileNotFoundError(f"Не найдено graphpart*.csv внутри {ROOT}")


def normalize_graphpart(gp):
    """
    Приводим graphpart к колонкам:
    protein_id, cluster
    """
    gp = gp.copy()

    if "protein_id" not in gp.columns:
        if "AC" in gp.columns:
            gp = gp.rename(columns={"AC": "protein_id"})
        elif "accession" in gp.columns:
            gp = gp.rename(columns={"accession": "protein_id"})
        else:
            raise ValueError(f"Не нашёл колонку protein id. Колонки: {gp.columns.tolist()}")

    if "cluster" not in gp.columns:
        raise ValueError(f"Не нашёл колонку cluster. Колонки: {gp.columns.tolist()}")

    gp["cluster"] = gp["cluster"].astype(int)

    return gp[["protein_id", "cluster"]]


all_summaries = {}

for gp_path in graphpart_files:
    gp = normalize_graphpart(pd.read_csv(gp_path))

    df = protein_counts.merge(gp, on="protein_id", how="inner")

    print("\n" + "=" * 80)
    print(gp_path)
    print("labeled proteins:", len(labeled))
    print("assigned proteins:", len(df))
    print("not assigned:", len(labeled) - len(df))

    summary = (
        df.groupby("cluster")
        .agg(
            n_proteins=("protein_id", "count"),
            total_residues=("total_residues", "sum"),
            OTHER_residues=("OTHER_residues", "sum"),
            PEPTIDE_residues=("PEPTIDE_residues", "sum"),
            PROPEP_residues=("PROPEP_residues", "sum"),
            n_with_peptide=("has_peptide", "sum"),
            n_with_propeptide=("has_propeptide", "sum"),
            n_peptide_features=("n_peptide_features", "sum"),
            n_propeptide_features=("n_propeptide_features", "sum"),
        )
        .reset_index()
        .sort_values("cluster")
    )

    for label in ["OTHER", "PEPTIDE", "PROPEP"]:
        summary[f"{label}_frac"] = (
            summary[f"{label}_residues"] / summary["total_residues"]
        )

    total = pd.DataFrame([{
        "cluster": "ALL",
        "n_proteins": df["protein_id"].nunique(),
        "total_residues": df["total_residues"].sum(),
        "OTHER_residues": df["OTHER_residues"].sum(),
        "PEPTIDE_residues": df["PEPTIDE_residues"].sum(),
        "PROPEP_residues": df["PROPEP_residues"].sum(),
        "n_with_peptide": df["has_peptide"].sum(),
        "n_with_propeptide": df["has_propeptide"].sum(),
        "n_peptide_features": df["n_peptide_features"].sum(),
        "n_propeptide_features": df["n_propeptide_features"].sum(),
    }])

    for label in ["OTHER", "PEPTIDE", "PROPEP"]:
        total[f"{label}_frac"] = (
            total[f"{label}_residues"] / total["total_residues"]
        )

    summary_with_total = pd.concat([summary, total], ignore_index=True)

    display_cols = [
        "cluster",
        "n_proteins",
        "total_residues",
        "OTHER_frac",
        "PEPTIDE_frac",
        "PROPEP_frac",
        "n_with_peptide",
        "n_with_propeptide",
        "n_peptide_features",
        "n_propeptide_features",
    ]

    print(summary_with_total[display_cols])

    out_path = gp_path.with_suffix(".class_balance.csv")
    summary_with_total.to_csv(out_path, index=False)

    print("saved:", out_path)

    all_summaries[gp_path.name] = summary_with_total

Found graphpart files:
 - ../data/uniprot_2022/graphpart_assignments.csv

../data/uniprot_2022/graphpart_assignments.csv
labeled proteins: 8449
assigned proteins: 7623
not assigned: 826
  cluster  n_proteins  total_residues  OTHER_frac  PEPTIDE_frac  PROPEP_frac  \
0       0        1504          421306    0.871352      0.059942     0.068705   
1       1        1526          510990    0.886070      0.043302     0.070628   
2       2        1425          265463    0.808293      0.075280     0.116427   
3       3        1630          283722    0.801105      0.082743     0.116152   
4       4        1538          365409    0.839065      0.075329     0.085605   
5     ALL        7623         1846890    0.849181      0.064090     0.086729   

   n_with_peptide  n_with_propeptide  n_peptide_features  \
0             600               1149                1666   
1             563               1257                1118   
2             775               1254                 977   
3            

In [19]:
from pathlib import Path
import pandas as pd

ROOT = Path("../data/uniprot_2026/processed_data")

features = pd.read_parquet(ROOT / "features_master.parquet")
proteins = pd.read_parquet(ROOT / "proteins_master.parquet")

features = features.merge(
    proteins[["accession", "length"]],
    on="accession",
    how="left",
    suffixes=("", "_protein"),
)

target = features[
    features["is_target_type"] &
    features["is_valid_for_training"]
].copy()

# PEPTIDE покрывает весь белок
full_length_peptides = target[
    (target["label"] == "PEPTIDE") &
    (target["start"] == 1) &
    (target["end"] == target["length_protein"])
]

print("Full-length PEPTIDE features:", len(full_length_peptides))
print("Proteins affected:", full_length_peptides["accession"].nunique())
print(full_length_peptides[["accession", "entry_name", "start", "end", "length", "length_protein", "description"]].head(20))

Full-length PEPTIDE features: 3817
Proteins affected: 3816
     accession   entry_name  start   end  length  length_protein  \
872     B3EWH2    AON_AZEFE    1.0  21.0    21.0              21   
1165    C0HJD9  K6TX1_ACTBE    1.0  17.0    17.0              17   
1180    C0HJQ2  KA192_BUTOC    1.0  31.0    31.0              31   
1215    C0HJY4  PON1A_ANOEM    1.0  16.0    16.0              16   
1220    C0HK44    LL3_LASLA    1.0  15.0    15.0              15   
1233    C0HL84   PNG1_PANCL    1.0  12.0    12.0              12   
1246    C0HL98   MAC1_MACFV    1.0  13.0    13.0              13   
1815    O00631  SARCO_HUMAN    1.0  31.0    31.0              31   
5039    P01521    CA1_CONMA    1.0  14.0    14.0              14   
5099    P01535   STX3_ANESU    1.0  27.0    27.0              27   
8030    P0C005   ANOP_ANOSM    1.0  10.0    10.0              10   
8052    P0C023   TX2B_MYRPI    1.0  23.0    23.0              23   
8115    P0C1Q4  MASTI_POLPI    1.0  14.0    14.0         